In [1]:
from pyspark.sql import SparkSession
from graphframes import GraphFrame

spark = (
    SparkSession.builder
    .appName("Macbook_Air_M3_Safe_GraphFrames")
    .master("local[4]")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.memoryOverhead", "512m")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.jars.packages", "io.graphframes:graphframes-spark4_2.13:0.12.2")
    .getOrCreate()
)

:: loading settings :: url = jar:file:/Users/quandkh/spark_learning/venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/quandkh/.ivy2.5.2/cache
The jars for the packages stored in: /Users/quandkh/.ivy2.5.2/jars
io.graphframes#graphframes-spark4_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d00a928f-596b-43cb-a732-64818f8cc171;1.0
	confs: [default]
	found io.graphframes#graphframes-spark4_2.13;0.12.2 in central
	found io.graphframes#graphframes-graphx-spark4_2.13;0.12.2 in central
	found org.apache.datasketches#datasketches-java;6.2.0 in central
	found org.apache.datasketches#datasketches-memory;3.0.2 in central
:: resolution report :: resolve 73ms :: artifacts dl 2ms
	:: modules in use:
	io.graphframes#graphframes-graphx-spark4_2.13;0.12.2 from central in [default]
	io.graphframes#graphframes-spark4_2.13;0.12.2 from central in [default]
	org.apache.datas

In [3]:
# 1. Tạo tập Đỉnh
vertices = spark.createDataFrame([
    ("1", "Alice", 28),
    ("2", "Bob", 32),
    ("3", "Charlie", 25)
], ["id", "name", "age"])

# 2. Tạo tập Cạnh
edges = spark.createDataFrame([
    ("1", "2", "friend"),
    ("2", "3", "follow"),
    ("3", "1", "friend")
], ["src", "dst", "relationship"])

# 3. Khởi tạo GraphFrame và tính toán
g = GraphFrame(vertices, edges)

print("--- Danh sách đỉnh ---")
g.vertices.show()

print("--- Số lượng kết nối (Degrees) ---")
g.degrees.show()

--- Danh sách đỉnh ---


+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 28|
|  2|    Bob| 32|
|  3|Charlie| 25|
+---+-------+---+

--- Số lượng kết nối (Degrees) ---
+---+------+
| id|degree|
+---+------+
|  1|     2|
|  2|     2|
|  3|     2|
+---+------+



In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [4]:
# Declare
motifs = g.find("(a)-[e1]->(b); (b)-[e2]->(c)")

#Show result
motifs.show(truncate=False)

# Filter
clean_motifs = (
    motifs.filter("a.id != c.id")
    .select("a.name", "e1.relationship", "b.name", "e2.relationship", "c.name")
)

clean_motifs.show()

+----------------+--------------+----------------+--------------+----------------+
|a               |e1            |b               |e2            |c               |
+----------------+--------------+----------------+--------------+----------------+
|{1, Alice, 28}  |{1, 2, friend}|{2, Bob, 32}    |{2, 3, follow}|{3, Charlie, 25}|
|{2, Bob, 32}    |{2, 3, follow}|{3, Charlie, 25}|{3, 1, friend}|{1, Alice, 28}  |
|{3, Charlie, 25}|{3, 1, friend}|{1, Alice, 28}  |{1, 2, friend}|{2, Bob, 32}    |
+----------------+--------------+----------------+--------------+----------------+

+-------+------------+-------+------------+-------+
|   name|relationship|   name|relationship|   name|
+-------+------------+-------+------------+-------+
|  Alice|      friend|    Bob|      follow|Charlie|
|    Bob|      follow|Charlie|      friend|  Alice|
|Charlie|      friend|  Alice|      friend|    Bob|
+-------+------------+-------+------------+-------+



In [5]:
#Find cycle

cycles = g.find("(a)-[e1]->(b); (b)-[e2]->(c); (c)-[e3]->(a)")

#Select columns to test
cycles.select("a.name", "b.name", "c.name").show()

+-------+-------+-------+
|   name|   name|   name|
+-------+-------+-------+
|  Alice|    Bob|Charlie|
|    Bob|Charlie|  Alice|
|Charlie|  Alice|    Bob|
+-------+-------+-------+



In [6]:
frenemies = (
    g.find("(a)-[e1]->(b); (b)-[e2]->(c)") 
    .filter("e1.relationship = 'friend' AND e2.relationship = 'follow'")
    .select("a.name", "b.name", "c.name")
)

frenemies.show()

+-----+----+-------+
| name|name|   name|
+-----+----+-------+
|Alice| Bob|Charlie|
+-----+----+-------+

